In [ ]:
# train.py
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
import joblib
import boto3

# Load data
X, y = load_iris(return_X_y=True)

# Train model
model = LogisticRegression(max_iter=200)
model.fit(X, y)

# Save model
joblib.dump(model, "model.joblib")

print("Model trained!")

# Upload to S3
bucket = "madhusudan-ml"   # change if needed
s3 = boto3.client("s3")

s3.upload_file("model.joblib", bucket, "model/model.joblib")

print("Model uploaded to S3!")

In [ ]:
import tarfile

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("model.joblib")
    tar.add("inference.py")   # 🔥 MUST

print("Tar file created!")

In [ ]:
import boto3

bucket = "madhusudan-ml"
s3 = boto3.client("s3")

s3.upload_file("model.tar.gz", bucket, "model/model.tar.gz")

print("Tar uploaded!")

In [ ]:
import sagemaker
from sagemaker.sklearn.model import SKLearnModel

role = sagemaker.get_execution_role()
session = sagemaker.Session()

model = SKLearnModel(
    model_data="s3://madhusudan-ml/model/model.tar.gz",
    role=role,
    entry_point="inference.py",
    source_dir=None,   # 🔥 FIX
    framework_version="1.0-1",
    sagemaker_session=session
)

predictor = model.deploy(
    instance_type="ml.t2.medium",
    initial_instance_count=1
)

print("Endpoint:", predictor.endpoint_name)

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

predictor.serializer = JSONSerializer()
predictor.deserializer = JSONDeserializer()

result = predictor.predict([[5.1, 3.5, 1.4, 0.2]])
print(result)

In [ ]:
#Create inference.py
#Seperate file
# inference.py
import joblib
import json
import os

def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))

def input_fn(request_body, request_content_type):
    return json.loads(request_body)

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, content_type):
    return json.dumps(prediction.tolist())

def predict_fn(input_data, model):
    try:
        prediction = model.predict(input_data)
        return prediction
    except Exception as e:
        print("Error in predict_fn:", e)
        raise e

def output_fn(prediction, content_type):
    try:
        return json.dumps(prediction.tolist())
    except Exception as e:
        print("Error in output_fn:", e)
        raise e

In [ ]:
#for testing in lambda funtion json
{
  "body": "{\"input\": [[5.1, 3.5, 1.4, 0.2]]}"
}

In [ ]:
to test api
curl -X POST "https://mhl2xa3qc4.execute-api.us-east-1.amazonaws.com/predict" -H "Content-Type: application/json" -d "{\"input\": [[5.1, 3.5, 1.4, 0.2]]}"

In [ ]:
incase of not allowed or policy problem add the following inline policy to sagemaker-predict-api-role-zzj53ksh or whatever

{
  "Version": "2012-10-17",
  "Id": "default",
  "Statement": [
    {
      "Sid": "FunctionURLAllowPublicAccess",
      "Effect": "Allow",
      "Principal": "*",
      "Action": "lambda:InvokeFunctionUrl",
      "Resource": "arn:aws:lambda:us-east-1:364453382858:function:sagemaker-predict-api",
      "Condition": {
        "StringEquals": {
          "lambda:FunctionUrlAuthType": "NONE"
        }
      }
    },
    {
      "Sid": "FunctionURLAllowInvokeAction",
      "Effect": "Allow",
      "Principal": "*",
      "Action": "lambda:InvokeFunction",
      "Resource": "arn:aws:lambda:us-east-1:364453382858:function:sagemaker-predict-api",
      "Condition": {
        "Bool": {
          "lambda:InvokedViaFunctionUrl": "true"
        }
      }
    },
    {
      "Sid": "67bfd43e-e634-5548-a42d-c4812ed11399",
      "Effect": "Allow",
      "Principal": {
        "Service": "apigateway.amazonaws.com"
      },
      "Action": "lambda:InvokeFunction",
      "Resource": "arn:aws:lambda:us-east-1:364453382858:function:sagemaker-predict-api",
      "Condition": {
        "ArnLike": {
          "AWS:SourceArn": "arn:aws:execute-api:us-east-1:364453382858:gysnivpm6f/*/*/predict"
        }
      }
    }
  ]
}